# SIH26054 — SOTA EDL Training on Kaggle

Clone this repo on Kaggle and train the EvidentialFaultClassifier + OOD detector.
No external datasets needed — data is physics-simulated.

**Prereqs**: Settings → Accelerator: `GPU T4 x2`, Internet: `ON`.

Replace `REPO_URL` in Cell 1 with your GitHub URL.

In [ ]:
# Cell 1 — Clone repo
import os, pathlib
REPO_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"
BRANCH = "master"

!rm -rf /kaggle/working/sih
!git clone {REPO_URL} /kaggle/working/sih
%cd /kaggle/working/sih
!git checkout {BRANCH}
!ls -la
!cat backend/config.py | head -40

In [ ]:
# Cell 2 — Install deps + verify GPU & simulator
%cd /kaggle/working/sih
!pip install -q filterpy onnx onnxruntime pyarrow tqdm
!pip install -q -e .

import torch
print("torch", torch.__version__, "cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

from backend.simulator.engine_simulator import EngineSimulator
sim = EngineSimulator(seed=0)
frame = sim.step()
print("Simulator OK — rpm:", round(frame.rpm,1), "cht:", [round(x,1) for x in frame.cht_c])

In [ ]:
# Cell 3 — Generate training data (physics-simulated)
# Quick smoke: ~1-2 min, 1k windows. Use for first verification.
# For full SOTA: comment quick and uncomment one of the full/default lines.
%cd /kaggle/working/sih
!python scripts/generate_all_data.py --quick
# !python scripts/generate_all_data.py              # default: 10 min missions, ~12k windows, 4-6 min
# !python scripts/generate_all_data.py --full       # full: 60 min missions, ~30k windows, 8-12 min

!ls -lh data/training/
!ls -lh data/models/residual_stats.npz
!cat data/training/generation_stats.json
import numpy as np, json, pathlib
d = np.load("data/training/windows.npz")
print("X", d["X"].shape, "y", d["y"].shape, "bincount", np.bincount(d["y"]))
from backend.config import FAULT_CLASSES
print("classes:", FAULT_CLASSES)

In [ ]:
# Cell 3ALT — (Optional) Skip generation if you uploaded windows.npz as Kaggle Dataset
# If you attached a dataset named sih-windows containing windows.npz + residual_stats.npz:
# !mkdir -p data/training data/models
# !cp /kaggle/input/sih-windows/windows.npz data/training/windows.npz
# !cp /kaggle/input/sih-windows/residual_stats.npz data/models/residual_stats.npz
# !ls -lh data/training/ data/models/
print("Skip — only run if you attached a pre-generated dataset")

In [ ]:
# Cell 4 — Train Evidential Model (SOTA)
%cd /kaggle/working/sih
# Quick: 3 epochs, ~30s
!python -m backend.ml.training.train_evidential --quick --num-workers 0

# Full SOTA: 60 epochs, early stopping, AMP on GPU
# !python -m backend.ml.training.train_evidential --epochs 60 --batch-size 64 --num-workers 0

# Tuned example:
# !python -m backend.ml.training.train_evidential --epochs 60 --lr 1e-3 --mixup 0.2 --scheduler cosine --num-workers 0

In [ ]:
# Cell 5 — Calibrate OOD detector (Mahalanobis)
# Auto-run at end of training; re-run standalone if needed
%cd /kaggle/working/sih
!python -m backend.ml.training.calibrate_ood --threshold 99.0
import numpy as np
stats = np.load("data/models/ood_stats.npz")
print("threshold", float(stats["threshold"]), "mean shape", stats["mean"].shape)
if "pca_components" in stats:
    print("PCA components", stats["pca_components"].shape)

In [ ]:
# Cell 6 — Validate & inspect
%cd /kaggle/working/sih
!python scripts/validate_models.py

import numpy as np
from backend.ml.inference import EvidentialModelInference
infer = EvidentialModelInference()
print("Inference loaded:", infer.loaded)
d = np.load("data/training/windows.npz")
X, y = d["X"], d["y"]
for i in [0, len(y)//3, len(y)//2, -1]:
    out = infer.infer(X[i])
    print(f"true={y[i]} ({infer.__class__.__name__}) pred={out['predicted_label']} epistemic={out['epistemic_uncertainty']:.3f} conf={out['confidence']:.3f} fault_prob={out['fault_probability']:.3f}")

# Plot training history
import json, pathlib
try:
    import matplotlib.pyplot as plt
    hist = json.loads(pathlib.Path("data/models/training_history.json").read_text())
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    axes[0].plot([h["train_loss"] for h in hist], label="train_loss")
    axes[0].plot([h["val_loss"] for h in hist], label="val_loss")
    axes[0].legend(); axes[0].set_title("Loss")
    axes[1].plot([h["val_acc"] for h in hist], label="val_acc")
    axes[1].plot([h["val_ece"] for h in hist], label="val_ece")
    axes[1].legend(); axes[1].set_title("Acc / ECE")
    plt.show()
except Exception as e:
    print("Plot skipped:", e)
    import json; print(json.dumps(hist[-1], indent=2) if 'hist' in locals() else "no hist")

In [ ]:
# Cell 7 — Save artifacts for download
%cd /kaggle/working/sih
!ls -lh data/models/
!cat data/models/val_metrics.json
!cat data/models/test_metrics.json
!zip -r /kaggle/working/sih_models.zip data/models/
print("Zip at /kaggle/working/sih_models.zip — Download via Kaggle Output panel (right sidebar)")
!ls -lh /kaggle/working/sih_models.zip
!echo "Files to commit back to repo:"
!ls -1 data/models/evidential_model.onnx data/models/evidential_model.pt data/models/ood_stats.npz data/models/residual_stats.npz 2>&1

### Optional: Push back to GitHub from Kaggle
Add a Kaggle Secret `GITHUB_TOKEN` (Settings → Secrets) then run:
```python
!git config --global user.email "kaggle@kaggle.com"
!git config --global user.name "kaggle"
!git add data/models/evidential_model.onnx data/models/ood_stats.npz data/models/residual_stats.npz data/models/training_history.json
!git commit -m "kaggle: add trained SOTA models"
!git push https://$GITHUB_TOKEN@github.com/<USER>/<REPO>.git HEAD:master
```
Or just download the zip and unzip locally into `data/models/`.